In [8]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
loaded_data = pd.read_csv('btc.csv')
data1 = loaded_data[["date","tf","USdollar","volume","big_wallets_whales","CPE_GAP"]].values
data2 = loaded_data[["price"]]


X = data1[:, 2:6]  # Columns: USdollar, volume, big_wallets_whales, CPE_GAP
y = data2.values.ravel()  # Price as target variable

print(f"X shape: {X.shape}")  # Should be (n_samples, 4)
print(f"y shape: {y.shape}")  # Should be (n_samples,)

# Now split and train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

alpha = model.intercept_
betas = model.coef_

print(f"Intercept (α): {alpha:.4f}")
print("Coefficients (β):")
for i, col in enumerate(["USdollar", "volume", "big_wallets_whales", "CPE_GAP"]):
    print(f"  {col}: {betas[i]:.4f}")

X shape: (30, 4)
y shape: (30,)
Intercept (α): -48574.0290
Coefficients (β):
  USdollar: 53.9134
  volume: 0.0559
  big_wallets_whales: 38.2894
  CPE_GAP: 2711.0622


In [17]:
x_train , x_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2 , random_state=42
)

In [18]:
model = LinearRegression()

model.fit(x_train, y_train)
alpha = model.intercept_
beta = model.coef_[0]



print(f"Intercept (α): {alpha:.2f}")
print(f"Slope (β): {beta:.2f}")


Intercept (α): -48574.03
Slope (β): 53.91


In [20]:
print(f"Intercept (α): {model.intercept_:.2f}")
print("Coefficients (β):")
feature_names = ["USdollar", "volume", "big_wallets_whales", "CPE_GAP"]
for i, (name, coef) in enumerate(zip(feature_names, model.coef_)):
    print(f"  {name}: {coef:.2f}")

Intercept (α): -48574.03
Coefficients (β):
  USdollar: 53.91
  volume: 0.06
  big_wallets_whales: 38.29
  CPE_GAP: 2711.06


In [21]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor

models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42),
    'SVM': SVR(kernel='rbf', C=1.0),
    'Neural Network': MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    print(f"{name}: R² = {r2:.4f}, MSE = {mse:.2f}")

Random Forest: R² = 0.8071, MSE = 137946.37
Gradient Boosting: R² = 0.8180, MSE = 130154.32
XGBoost: R² = 0.1442, MSE = 611945.09
SVM: R² = -0.7722, MSE = 1267214.53
Neural Network: R² = -6.7459, MSE = 5538850.94


In [22]:
# Create new features
loaded_data['price_lag1'] = loaded_data['price'].shift(1)
loaded_data['volume_change'] = loaded_data['volume'].pct_change()
loaded_data['usd_volume_interaction'] = loaded_data['USdollar'] * loaded_data['volume']
loaded_data['whale_ratio'] = loaded_data['big_wallets_whales'] / loaded_data['big_wallets_whales'].rolling(7).mean()

# Remove NaN values
loaded_data = loaded_data.dropna()

# Updated features
feature_columns = ["USdollar", "volume", "big_wallets_whales", "CPE_GAP", 
                   "price_lag1", "volume_change", "usd_volume_interaction", "whale_ratio"]

In [23]:
from sklearn.model_selection import GridSearchCV

# For Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='r2')
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best R²: {grid_search.best_score_:.4f}")

Best parameters: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
Best R²: 0.5398


In [24]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Create time-based features
loaded_data['day_of_week'] = pd.to_datetime(loaded_data['date']).dt.dayofweek
loaded_data['month'] = pd.to_datetime(loaded_data['date']).dt.month
loaded_data['price_rolling_mean_7'] = loaded_data['price'].rolling(7).mean()

# Pipeline with scaling
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42))
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=None, device=None,
                              early_stopping_rounds=None,
                              enable_categorical=False, eval_metric=None,
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.1,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=200, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [26]:
from sklearn.ensemble import VotingRegressor

# Combine multiple models
ensemble = VotingRegressor([
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
    ('xgb', XGBRegressor(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingRegressor(n_estimators=100, random_state=42))
])

ensemble.fit(X_train, y_train)
y_pred_ensemble = ensemble.predict(X_test)

In [27]:
def optimize_btc_model(loaded_data):
    # Feature engineering
    loaded_data['price_lag1'] = loaded_data['price'].shift(1)
    loaded_data['volume_change'] = loaded_data['volume'].pct_change()
    loaded_data['price_rolling_mean_7'] = loaded_data['price'].rolling(7).mean()
    loaded_data = loaded_data.dropna()
    
    # Features and target
    feature_columns = ["USdollar", "volume", "big_wallets_whales", "CPE_GAP", 
                       "price_lag1", "volume_change", "price_rolling_mean_7"]
    X = loaded_data[feature_columns]
    y = loaded_data['price']
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Try multiple models
    models = {
        'XGBoost': XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42),
        'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
        'Gradient Boost': GradientBoostingRegressor(n_estimators=200, random_state=42)
    }
    
    best_model = None
    best_r2 = -float('inf')
    
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        
        print(f"{name}: R² = {r2:.4f}, MSE = {mse:.2f}")
        
        if r2 > best_r2:
            best_r2 = r2
            best_model = model
    
    return best_model, best_r2

# Run optimization
best_model, best_r2 = optimize_btc_model(loaded_data)
print(f"\nBest model R²: {best_r2:.4f}")

XGBoost: R² = 0.6089, MSE = 203600.11
Random Forest: R² = 0.7873, MSE = 110710.25
Gradient Boost: R² = 0.3558, MSE = 335326.26

Best model R²: 0.7873


In [28]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best R²: {grid_search.best_score_:.4f}")
print(f"Best params: {grid_search.best_params_}")

Best R²: 0.5398
Best params: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}


In [29]:
# Add these features to your existing ones
loaded_data['price_momentum'] = loaded_data['price'] / loaded_data['price'].shift(5) - 1
loaded_data['volatility_7'] = loaded_data['price'].rolling(7).std()
loaded_data['usd_whale_interaction'] = loaded_data['USdollar'] * loaded_data['big_wallets_whales']
loaded_data['volume_ma_ratio'] = loaded_data['volume'] / loaded_data['volume'].rolling(7).mean()

# Remove any remaining NaN
loaded_data = loaded_data.dropna()

In [31]:
# Check which features are most important
best_rf = grid_search.best_estimator_

# Get the actual number of features from the model
n_features = best_rf.n_features_in_

# Use the correct feature names based on what was actually trained
# The grid_search was trained on X_train which has 4 features from cell 2
actual_features = ["USdollar", "volume", "big_wallets_whales", "CPE_GAP"]

feature_importance = pd.DataFrame({
    'feature': actual_features,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance:")
print(feature_importance)

Feature Importance:
              feature  importance
1              volume    0.345217
0            USdollar    0.332124
2  big_wallets_whales    0.250898
3             CPE_GAP    0.071761


In [32]:
from sklearn.ensemble import VotingRegressor

# Combine top performers
ensemble = VotingRegressor([
    ('rf', RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)),
    ('xgb', XGBRegressor(n_estimators=150, learning_rate=0.1, random_state=42))
])

ensemble.fit(X_train, y_train)
y_pred_ensemble = ensemble.predict(X_test)
r2_ensemble = r2_score(y_test, y_pred_ensemble)
print(f"Ensemble R²: {r2_ensemble:.4f}")

Ensemble R²: 0.5426


In [33]:
from sklearn.feature_selection import SelectFromModel

# Use Random Forest for feature selection
selector = SelectFromModel(best_rf, prefit=True)
X_selected = selector.transform(X)

print(f"Original features: {X.shape[1]}")
print(f"Selected features: {X_selected.shape[1]}")

Original features: 4
Selected features: 3
